# Floor-plan SegFormer fine-tuning

Select **Runtime → Change runtime type → T4 GPU**. Upload the source-only project archive generated locally with `python scripts/package_source.py`. The dataset and checkpoint retain upstream non-commercial terms. No paid compute is required by this notebook.


In [ ]:
import subprocess, sys
print(subprocess.check_output(["nvidia-smi"], text=True))
from google.colab import files
uploaded = files.upload()


In [ ]:
from pathlib import Path
import zipfile, io
ROOT = Path("/content/floorplan_project")
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
    z.extractall(ROOT)


## Train and evaluate

The job installs the project, verifies the official dataset checksum, preprocesses the official splits, fine-tunes all model parameters, selects by validation foreground mIoU, and evaluates the untouched test split. Checkpoints and optimizer state are saved after each epoch in the runtime. Download a backup before disconnecting; runtime files are temporary.


In [ ]:
with open("/content/floorplan_job.log", "a", buffering=1) as log:
    job = subprocess.Popen([sys.executable, "-u", "scripts/cloud_run.py"], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    import time
    while job.poll() is None:
        time.sleep(30)
        print(Path("/content/floorplan_job.log").read_text()[-1000:], flush=True)
    if job.returncode:
        raise RuntimeError(Path("/content/floorplan_job.log").read_text()[-5000:])


## Download local artifacts

Extract the returned zip into the local project. It contains the best weights, initial checkpoint, training logs, measured evaluation, and example JSON outputs. It excludes the full dataset.


In [ ]:
files.download("/content/floorplan_trained_artifacts.zip")
